# 🏛️ Universal Screenplay AST Generator
### Automated, Deterministic & Strictly-Typed Ingestion for Any Microdrama Series

This notebook converts **any folder of raw screenplay text files** into a canonical, strictly-typed **JSON Abstract Syntax Tree (AST)**.

---

### Core Capabilities:
1. **Universal Series Ingestion**: Point to *any* folder containing screenplay `.txt` files—regardless of naming conventions (`ep01.txt`, `episode-001.txt`, `1.txt`, etc.).
2. **Strict Type Architecture**: Native Python 3.13 `@dataclass(slots=True, frozen=True)` and typed Enums with mechanical validation invariants.
3. **Exact Integer Centisecond Math**: Replaces floating-point timestamp drift with exact integer centiseconds ($1\text{s} = 100\text{cs}$).
4. **Universal Grammar Parsing**:
   - Case-insensitive scene delimiters (`SCENE HEADING:`, `Scene heading:`).
   - Decomposes dialogue tags into speaker, audio modifier (`O.S.`, `V.O.`, `Off-screen`), and acting/delivery tone (`Cold, Commanding`).
   - Categorizes all 4 screenplay element types: `CAMERA`, `ACTION`, `SOUND FX`, and `DIALOGUE`.
5. **Standalone Reusability**: Outputs a standardized JSON AST file that downstream LLM extraction, synthesis, and ranking modules can load directly.


<!-- ## 1. Strictly-Typed AST Data Structures -->
We define the immutable data contract for atomic screenplay elements, scenes, headers, and entire series corpora.


In [1]:
from enum import Enum
from dataclasses import dataclass, asdict
from typing import Optional, List, Tuple
import re
# import os
import json
from pathlib import Path
import pandas as pd

class ElementType(str, Enum):
    CAMERA = "CAMERA"
    ACTION = "ACTION"
    SOUND_FX = "SOUND FX"
    DIALOGUE = "DIALOGUE"


@dataclass(slots=True, frozen=True)
class DialogueMetadata:
    """Strongly-typed metadata for dialogue elements."""
    speaker: str
    modifier: Optional[str] = None   # e.g., 'O.S.', 'V.O.', 'Off-screen'
    delivery: Optional[str] = None   # e.g., 'Cold, Commanding', 'Grave, Alert'
    raw_tag: str = ""

    def __post_init__(self):
        if not self.speaker.strip():
            raise ValueError("Speaker name cannot be empty in DialogueMetadata")


@dataclass(slots=True, frozen=True)
class ScreenplayElement:
    """Atomic screenplay unit with exact integer centisecond timing."""
    element_id: str                      # Format: EP{ep:02d}_EL{el:03d}
    episode_num: int
    scene_num: int
    element_type: ElementType
    start_raw: str                       # e.g., '00:11.30'
    end_raw: str                         # e.g., '00:13.10'
    start_cs: int                        # integer centiseconds (1s = 100cs)
    end_cs: int
    duration_cs: int                     # end_cs - start_cs
    duration_seconds: float              # duration_cs / 100.0
    content: str                         # verbatim dialogue or action text
    dialogue_meta: Optional[DialogueMetadata] = None
    raw_tag: str = ""

    def __post_init__(self):
        if not isinstance(self.element_type, ElementType):
            raise TypeError(f"Invalid element_type: {self.element_type}")
        if self.start_cs < 0:
            raise ValueError(f"start_cs must be non-negative: {self.start_cs}")
        if self.end_cs <= self.start_cs:
            raise ValueError(f"end_cs ({self.end_cs}) must be strictly greater than start_cs ({self.start_cs})")
        if self.duration_cs != (self.end_cs - self.start_cs):
            raise ValueError("duration_cs does not match (end_cs - start_cs)")
        if not self.content.strip():
            raise ValueError(f"Element {self.element_id} has empty content")


@dataclass(slots=True, frozen=True)
class Scene:
    """Scene boundary containing chronological screenplay elements."""
    episode_num: int
    scene_num: int
    heading: str
    start_raw: str
    end_raw: str
    start_cs: int
    end_cs: int
    duration_cs: int
    elements: Tuple[ScreenplayElement, ...]

    def __post_init__(self):
        if self.end_cs <= self.start_cs:
            raise ValueError(f"Scene end_cs ({self.end_cs}) must exceed start_cs ({self.start_cs})")


@dataclass(slots=True, frozen=True)
class EpisodeHeader:
    """Metadata header block for an episode."""
    episode_num: int
    raw_episode_title: str
    logline: str
    continuity_notes: str


@dataclass(slots=True, frozen=True)
class Episode:
    """Structured episode containing header, scenes, and ordered elements."""
    header: EpisodeHeader
    scenes: Tuple[Scene, ...]
    elements: Tuple[ScreenplayElement, ...]
    total_elements: int
    total_duration_cs: int

    @property
    def episode_num(self) -> int:
        return self.header.episode_num

    @property
    def total_duration_seconds(self) -> float:
        return self.total_duration_cs / 100.0


@dataclass(slots=True, frozen=True)
class ValidationAudit:
    """Validation audit record verifying mechanical integrity."""
    episode_num: int
    filename: str
    scene_count: int
    camera_count: int
    action_count: int
    sound_fx_count: int
    dialogue_count: int
    total_elements: int
    timestamp_errors: int
    containment_errors: int
    is_valid: bool
    errors: Tuple[str, ...] = ()

print("[OK] Strict Type Schema loaded successfully.")


[OK] Strict Type Schema loaded successfully.


This class handles all parsing logic deterministically:
- Natural numeric sorting for any file naming scheme (`ep01.txt`, `episode-001.txt`, `1.txt`).
- Multi-format timestamp converter (`MM:SS.ss` or `HH:MM:SS.ss` $\rightarrow$ integer centiseconds).
- Dynamic tag decomposition (extracts speaker, audio modifier, and delivery tone without hardcoded character names).
- Case-tolerant scene delimiters and metadata header extractors.


In [2]:
class UniversalScreenplayParser:
    """
    Production-grade parser for any microdrama series screenplay corpus.
    """
    def __init__(self):
        # Supports MM:SS.ss or HH:MM:SS.ss
        self.ts_pattern = re.compile(r"(?:(\d{1,2}):)?(\d{2}):(\d{2})\.(\d{2})")
        
        # Element block pattern
        self.element_pattern = re.compile(
            r"^\*\*(?P<tag>.+?):\*\*\s*\n"
            r"Start:\s*\((?P<start>[^)]+)\)\s*\|\s*End:\s*\((?P<end>[^)]+)\)\s*\n"
            r"(?P<content>.*?)(?=(?:\n\*\*.+?:\*\*\s*\nStart:|\nSCENE\s+\d+|\Z))",
            re.MULTILINE | re.DOTALL | re.IGNORECASE
        )
        
        # Scene block pattern
        self.scene_pattern = re.compile(
            r"^(SCENE\s+(?P<num>\d+))\s*\n"
            r"Scene heading:\s*(?P<heading>.+?)\s*\n"
            r"Scene start:\s*(?P<start>[0-9]{2}:[0-9]{2}\.[0-9]{2})\s*\n"
            r"Scene end:\s*(?P<end>[0-9]{2}:[0-9]{2}\.[0-9]{2})\s*\n"
            r"Actions and dialogue:",
            re.MULTILINE | re.IGNORECASE
        )

    def time_to_centiseconds(self, ts_str: str) -> int:
        """Converts any MM:SS.ss or HH:MM:SS.ss string into integer centiseconds."""
        m = self.ts_pattern.search(ts_str.strip())
        if not m:
            raise ValueError(f"Malformed timestamp: '{ts_str}'")
        hh = int(m.group(1)) if m.group(1) else 0
        mm = int(m.group(2))
        ss = int(m.group(3))
        cs = int(m.group(4))
        return hh * 360000 + mm * 6000 + ss * 100 + cs

    def decompose_tag(self, raw_tag_str: str) -> Tuple[ElementType, Optional[DialogueMetadata]]:
        """Decomposes raw tag into ElementType and structured DialogueMetadata."""
        tag = raw_tag_str.strip()
        tag_upper = tag.upper()
        
        if tag_upper == "CAMERA":
            return ElementType.CAMERA, None
        if tag_upper == "ACTION":
            return ElementType.ACTION, None
        if "SOUND FX" in tag_upper:
            return ElementType.SOUND_FX, None
            
        # Dialogue: extract speaker and optional parenthetical notes
        parens = re.findall(r"\((.*?)\)", tag)
        first_paren_idx = tag.find("(")
        speaker = tag[:first_paren_idx].strip() if first_paren_idx != -1 else tag.strip()
        
        modifier = None
        delivery = None
        if len(parens) == 1:
            p = parens[0].strip()
            if p.upper() in {"O.S.", "V.O.", "OFF-SCREEN", "OFF SCREEN"}:
                modifier = p
            else:
                delivery = p
        elif len(parens) >= 2:
            modifier = parens[0].strip()
            delivery = " | ".join(p.strip() for p in parens[1:])
            
        meta = DialogueMetadata(
            speaker=speaker,
            modifier=modifier,
            delivery=delivery,
            raw_tag=tag
        )
        return ElementType.DIALOGUE, meta

    def parse_screenplay_file(self, file_path: Path, episode_idx: int) -> Episode:
        """Parses a single screenplay file into a typed Episode object."""
        text = file_path.read_text(encoding="utf-8")
        
        # 1. Parse Headers
        ep_num_m = re.search(r"^EPISODE NUMBER:\s*(?:Episode\s*)?(\d+)", text, re.M | re.I)
        ep_num = int(ep_num_m.group(1)) if ep_num_m else episode_idx
        
        title_m = re.search(r"^EPISODE NUMBER:\s*(.+)$", text, re.M | re.I)
        raw_title = title_m.group(1).strip() if title_m else f"Episode {ep_num}"
        
        logline_m = re.search(r"^(?:EPISODE\s+)?LOGLINE:\s*(.+)$", text, re.M | re.I)
        logline = logline_m.group(1).strip() if logline_m else ""
        
        notes_m = re.search(r"^(?:CONTINUITY\s+)?NOTES:\s*(.+)$", text, re.M | re.I)
        notes = notes_m.group(1).strip() if notes_m else ""
        
        header = EpisodeHeader(
            episode_num=ep_num,
            raw_episode_title=raw_title,
            logline=logline,
            continuity_notes=notes
        )
        
        # 2. Extract script body
        split_marker = "--- SCRIPT START ---"
        body_idx = text.find(split_marker)
        body_text = text[body_idx + len(split_marker):] if body_idx != -1 else text
        
        # 3. Locate Scenes
        scene_matches = list(self.scene_pattern.finditer(body_text))
        if not scene_matches:
            # Fallback single scene for monolithic scripts
            scene_spans = [(0, len(body_text), 1, "SCENE 01", "00:00.00", "00:00.00")]
        else:
            scene_spans = []
            for i, sm in enumerate(scene_matches):
                s_pos = sm.end()
                e_pos = scene_matches[i + 1].start() if i + 1 < len(scene_matches) else len(body_text)
                scene_spans.append((
                    s_pos,
                    e_pos,
                    int(sm.group("num")),
                    sm.group("heading").strip(),
                    sm.group("start").strip(),
                    sm.group("end").strip()
                ))
                
        # 4. Parse elements inside each scene
        parsed_scenes = []
        all_elements = []
        element_counter = 0
        
        for s_pos, e_pos, sc_num, heading, sc_start_raw, sc_end_raw in scene_spans:
            scene_body = body_text[s_pos:e_pos]
            sc_start_cs = self.time_to_centiseconds(sc_start_raw)
            sc_end_cs = self.time_to_centiseconds(sc_end_raw)
            
            scene_elements = []
            for em in self.element_pattern.finditer(scene_body):
                element_counter += 1
                el_id = f"EP{ep_num:02d}_EL{element_counter:03d}"
                raw_tag = em.group("tag").strip()
                start_raw = em.group("start").strip()
                end_raw = em.group("end").strip()
                content = em.group("content").strip()
                
                start_cs = self.time_to_centiseconds(start_raw)
                end_cs = self.time_to_centiseconds(end_raw)
                duration_cs = end_cs - start_cs
                
                el_type, dlg_meta = self.decompose_tag(raw_tag)
                
                elem = ScreenplayElement(
                    element_id=el_id,
                    episode_num=ep_num,
                    scene_num=sc_num,
                    element_type=el_type,
                    start_raw=start_raw,
                    end_raw=end_raw,
                    start_cs=start_cs,
                    end_cs=end_cs,
                    duration_cs=duration_cs,
                    duration_seconds=duration_cs / 100.0,
                    content=content,
                    dialogue_meta=dlg_meta,
                    raw_tag=raw_tag
                )
                scene_elements.append(elem)
                all_elements.append(elem)
                
            # Scene end fallback if scene end was zero
            if sc_end_cs <= sc_start_cs and scene_elements:
                sc_end_cs = scene_elements[-1].end_cs
                sc_end_raw = scene_elements[-1].end_raw
                
            scene = Scene(
                episode_num=ep_num,
                scene_num=sc_num,
                heading=heading,
                start_raw=sc_start_raw,
                end_raw=sc_end_raw,
                start_cs=sc_start_cs,
                end_cs=sc_end_cs,
                duration_cs=sc_end_cs - sc_start_cs,
                elements=tuple(scene_elements)
            )
            parsed_scenes.append(scene)
            
        total_duration_cs = sum(s.duration_cs for s in parsed_scenes) if parsed_scenes else (all_elements[-1].end_cs if all_elements else 0)
        
        return Episode(
            header=header,
            scenes=tuple(parsed_scenes),
            elements=tuple(all_elements),
            total_elements=len(all_elements),
            total_duration_cs=total_duration_cs
        )

print("[OK] UniversalScreenplayParser engine ready.")


[OK] UniversalScreenplayParser engine ready.


## Controller
Set the target directory and destination path below to generate an AST for **any** microdrama series.


In [3]:
INPUT_DIRECTORY = Path("input_scripts")                # Target directory with .txt screenplays
OUTPUT_AST_PATH = Path("01_structured_episodes.json")  # Canonical output JSON path
SERIES_TITLE    = "I Married a Broken Soldier. He was a Duke"  # Leave None to infer from directory name

# Auto-discover screenplay files with natural numeric ordering
def discover_screenplay_files(folder: Path) -> List[Path]:
    """Discovers and naturally sorts screenplay text files."""
    txt_files = list(folder.glob("*.txt"))
    # Natural sort based on any numbers in the filename
    def extract_num(p: Path) -> int:
        digits = re.findall(r"\d+", p.stem)
        return int(digits[0]) if digits else 999999
    return sorted(txt_files, key=extract_num)

discovered_files = discover_screenplay_files(INPUT_DIRECTORY)
print(f"Target Directory : {INPUT_DIRECTORY.resolve()}")
print(f"Files Discovered : {len(discovered_files)} files found")
if discovered_files:
    print(f"Sample range     : {discovered_files[0].name} ... {discovered_files[-1].name}")


Target Directory : C:\Users\Anirudh Singh\OneDrive\Documents\Kuku FM\input_scripts
Files Discovered : 50 files found
Sample range     : ep01.txt ... ep50.txt


## 4. Execute Full Ingestion & Mechanical Validation Audit
Run the parser across all discovered episodes and verify that **0 timestamp errors** occur.


In [4]:
parser = UniversalScreenplayParser()
parsed_episodes: List[Episode] = []
audit_records: List[ValidationAudit] = []

print(f"Starting deterministic parsing of {len(discovered_files)} files...")

for idx, file_path in enumerate(discovered_files, start=1):
    ep = parser.parse_screenplay_file(file_path, episode_idx=idx)
    parsed_episodes.append(ep)
    
    # Audit checks
    cam_c = sum(1 for e in ep.elements if e.element_type == ElementType.CAMERA)
    act_c = sum(1 for e in ep.elements if e.element_type == ElementType.ACTION)
    snd_c = sum(1 for e in ep.elements if e.element_type == ElementType.SOUND_FX)
    dlg_c = sum(1 for e in ep.elements if e.element_type == ElementType.DIALOGUE)
    
    t_errs = 0
    c_errs = 0
    errors = []
    
    for sc in ep.scenes:
        if sc.start_cs >= sc.end_cs:
            t_errs += 1
            errors.append(f"Scene {sc.scene_num} invalid window: {sc.start_cs} >= {sc.end_cs}")
        for el in sc.elements:
            if el.start_cs >= el.end_cs:
                t_errs += 1
                errors.append(f"Element {el.element_id} monotonic failure: {el.start_cs} >= {el.end_cs}")
            if el.start_cs < sc.start_cs or el.end_cs > (sc.end_cs + 100):
                c_errs += 1
                
    audit = ValidationAudit(
        episode_num=ep.episode_num,
        filename=file_path.name,
        scene_count=len(ep.scenes),
        camera_count=cam_c,
        action_count=act_c,
        sound_fx_count=snd_c,
        dialogue_count=dlg_c,
        total_elements=ep.total_elements,
        timestamp_errors=t_errs,
        containment_errors=c_errs,
        is_valid=(len(errors) == 0 and t_errs == 0),
        errors=tuple(errors)
    )
    audit_records.append(audit)

# Create summary audit DataFrame
df_audit = pd.DataFrame([{
    "Episode": f"Ep {a.episode_num:02d}",
    "File": a.filename,
    "Scenes": a.scene_count,
    "CAMERA": a.camera_count,
    "ACTION": a.action_count,
    "SOUND FX": a.sound_fx_count,
    "DIALOGUE": a.dialogue_count,
    "Total Elements": a.total_elements,
    "Timestamp Errors": a.timestamp_errors,
    "Status": "PASSED" if a.is_valid else "FAILED"
} for a in audit_records])

total_time_failures = sum(a.timestamp_errors for a in audit_records)
print("=" * 60)
print(f"AUDIT SUMMARY FOR: {SERIES_TITLE or INPUT_DIRECTORY.name}")
print("=" * 60)
print(f"Total Episodes Processed  : {len(parsed_episodes):5d}")
print(f"Total Scenes Detected     : {sum(len(ep.scenes) for ep in parsed_episodes):5d}")
print(f"Total Timed Elements      : {sum(ep.total_elements for ep in parsed_episodes):5d}")
print(f"  - CAMERA Elements       : {sum(a.camera_count for a in audit_records):5d}")
print(f"  - ACTION Elements       : {sum(a.action_count for a in audit_records):5d}")
print(f"  - SOUND FX Elements     : {sum(a.sound_fx_count for a in audit_records):5d}")
print(f"  - DIALOGUE Elements     : {sum(a.dialogue_count for a in audit_records):5d}")
print(f"Timestamp Monotonic Errors: {total_time_failures:5d}")
print("=" * 60)

assert total_time_failures == 0, "Discovered timestamp monotonicity errors!"

# Display first 5 and last 5 episodes in audit table
print("\nAudit Table Preview:")
print(pd.concat([df_audit.head(5), df_audit.tail(5)]).to_string(index=False))


Starting deterministic parsing of 50 files...
AUDIT SUMMARY FOR: I Married a Broken Soldier. He was a Duke
Total Episodes Processed  :    50
Total Scenes Detected     :   100
Total Timed Elements      :  3727
  - CAMERA Elements       :  1754
  - ACTION Elements       :   372
  - SOUND FX Elements     :   114
  - DIALOGUE Elements     :  1487
Timestamp Monotonic Errors:     0

Audit Table Preview:
Episode     File  Scenes  CAMERA  ACTION  SOUND FX  DIALOGUE  Total Elements  Timestamp Errors Status
  Ep 01 ep01.txt       1      50       7         1        37              95                 0 PASSED
  Ep 02 ep02.txt       1      39       4         0        31              74                 0 PASSED
  Ep 03 ep03.txt       3      61       8         3        46             118                 0 PASSED
  Ep 04 ep04.txt       1      59       9         2        40             110                 0 PASSED
  Ep 05 ep05.txt       1      25       6         2        29              62             

## 5. Export Canonical JSON AST
We serialize the validated corpus into the standardized JSON AST format, complete with element-level centisecond math and dialogue metadata.


In [5]:
OUTPUT_AST_PATH.parent.mkdir(parents=True, exist_ok=True)

canonical_ast = {
    "corpus_title": SERIES_TITLE or INPUT_DIRECTORY.name.replace("_", " ").title(),
    "total_episodes": len(parsed_episodes),
    "total_elements": sum(ep.total_elements for ep in parsed_episodes),
    "element_distribution": {
        "CAMERA": sum(a.camera_count for a in audit_records),
        "ACTION": sum(a.action_count for a in audit_records),
        "SOUND_FX": sum(a.sound_fx_count for a in audit_records),
        "DIALOGUE": sum(a.dialogue_count for a in audit_records)
    },
    "episodes": [
        {
            "episode_num": ep.episode_num,
            "header": asdict(ep.header),
            "scenes": [
                {
                    "scene_num": sc.scene_num,
                    "heading": sc.heading,
                    "start_cs": sc.start_cs,
                    "end_cs": sc.end_cs,
                    "start_raw": sc.start_raw,
                    "end_raw": sc.end_raw,
                    "duration_cs": sc.duration_cs,
                    "duration_seconds": sc.duration_cs / 100.0,
                    "elements": [asdict(e) for e in sc.elements]
                }
                for sc in ep.scenes
            ]
        }
        for ep in parsed_episodes
    ]
}

OUTPUT_AST_PATH.write_text(json.dumps(canonical_ast, indent=2), encoding="utf-8")
file_size_kb = OUTPUT_AST_PATH.stat().st_size / 1024
print(f"[OK] Successfully exported AST to: {OUTPUT_AST_PATH.resolve()}")
print(f"     Total Size: {file_size_kb:.1f} KB")


[OK] Successfully exported AST to: C:\Users\Anirudh Singh\OneDrive\Documents\Kuku FM\01_structured_episodes.json
     Total Size: 2478.0 KB


## 6. Interactive AST Inspector
Inspect any episode, scene, or element from the generated AST database.


In [6]:
def inspect_ast_episode(ast_data: dict, episode_number: int = 1):
    """Pretty-prints an episode and its elements from the structured AST."""
    ep_data = next((e for e in ast_data["episodes"] if e["episode_num"] == episode_number), None)
    if not ep_data:
        print(f"Episode {episode_number} not found in AST.")
        return
        
    hdr = ep_data["header"]
    print("=" * 75)
    print(f"EPISODE {hdr['episode_num']:02d}: {hdr['raw_episode_title']}")
    print("=" * 75)
    print(f"Logline         : {hdr['logline']}")
    print(f"Continuity Notes: {hdr['continuity_notes']}")
    print(f"Scenes Count    : {len(ep_data['scenes'])}")
    
    for sc in ep_data["scenes"]:
        print(f"\n  [SCENE {sc['scene_num']:02d}] {sc['heading']} [{sc['start_raw']} -> {sc['end_raw']}] ({sc['duration_seconds']:.2f}s)")
        print(f"  Total Elements: {len(sc['elements'])}")
        
        # Sample first 3 elements of each scene
        for el in sc["elements"][:3]:
            el_type = el["element_type"]
            print(f"    - {el['element_id']} [{el_type:8s}] [{el['start_raw']}->{el['end_raw']}]: {el['content'][:75]}...")

# Inspect Episode 1
inspect_ast_episode(canonical_ast, episode_number=1)


EPISODE 01: Episode 1
Logline         : Lady Juliana arrives at the cathedral in her late mother's bridal gown, only to suffer the ultimate betrayal when her fiancé Gerald abandons her at the altar for her cruel sister Clarissa with her family's full blessing.
Continuity Notes: First episode of the series. Establishes Juliana's humiliation and abandonment by her betrothed, Gerald, and her noble family in favor of her sister, Clarissa.
Scenes Count    : 1

  [SCENE 01] INT. CATHEDRAL – DAY [00:00.00 -> 02:18.00] (138.00s)
  Total Elements: 95
    - EP01_EL001 [ElementType.CAMERA] [00:00.00->00:04.80]: Symmetrical framing on the heavy church doors as they slowly part, pushing ...
    - EP01_EL002 [ElementType.ACTION] [00:01.00->00:04.80]: Juliana pauses at the threshold, her expression bright with joyful anticipa...
    - EP01_EL003 [ElementType.CAMERA] [00:04.80->00:06.80]: High wide tracking shot following behind Juliana as she steps into the gran...
